In [9]:
import numpy as np

def accuracys1(gt1, gt2, p1, p2):
    gt1_max = np.argmax(gt1, axis=1)
    gt2_max = np.argmax(gt2, axis=1)
    p1_max = np.argmax(p1, axis=1)
    p2_max = np.argmax(p2, axis=1)

    at_least_one = (
        (p1_max == gt1_max)
        | (p1_max == gt2_max)
        | (p2_max == gt1_max)
        | (p2_max == gt2_max)
    ).astype(int)

    pred_pairs = np.sort(np.stack([p1_max, p2_max], axis=1), axis=1)
    y_pairs = np.sort(np.stack([gt1_max, gt2_max], axis=1), axis=1)
    both = np.all(pred_pairs == y_pairs, axis=1).astype(int)

    acc_at_least_one = np.count_nonzero(at_least_one) / len(at_least_one)
    acc_both = np.count_nonzero(both) / len(both)

    return round(acc_at_least_one, 2), round(acc_both, 2)

def accuracys2(gt1, gt2, p1, p2):
        gt1_max = np.argmax(gt1, axis=1)
        gt2_max = np.argmax(gt2, axis=1)
        p1_max = np.argmax(p1, axis=1)
        p2_max = np.argmax(p2, axis=1)

        at_least_one = (
            (p1_max == gt1_max)
            | (p1_max == gt2_max)
            | (p2_max == gt1_max)
            | (p2_max == gt2_max)
        ).astype(int)

        pred_pairs = np.sort(np.stack([p1_max, p2_max], axis=1), axis=1)
        y_pairs = np.sort(np.stack([gt1_max, gt2_max], axis=1), axis=1)
        both = np.all(pred_pairs == y_pairs, axis=1).astype(int)

        acc_at_least_one = np.count_nonzero(at_least_one) / len(at_least_one)
        acc_both = np.count_nonzero(both) / len(both)

        return round(acc_at_least_one, 5), round(acc_both, 5)

In [6]:
gt1 = np.array([
    [1,0,0],
    [0,1,0],
    [0,1,0],
    [0,0,1],
    [1,0,0],
])

gt2 = np.array([
    [0,1,0],
    [0,0,1],
    [1,0,0],
    [0,1,0],
    [0,0,1],
])

p1 = np.array([
    [1,0,0],  # hit
    [0,1,0],  # hit
    [0,0,1],  # hit gt2
    [0,1,0],  # hit gt2
    [0,0,1],  # hit
])

p2 = np.array([
    [0,0,1],  # miss
    [1,0,0],  # hit gt2
    [0,1,0],  # hit
    [0,0,1],  # miss
    [1,0,0],  # hit
])



In [7]:
accuracys1(gt1=gt1,gt2=gt2,p1=p1,p2=p2)

(1.0, 0.4)

In [10]:
accuracys2(gt1=gt1,gt2=gt2,p1=p1,p2=p2)

(1.0, 0.4)

In [12]:
import tensorflow as tf
from keras.layers  import Layer

class Sampling(Layer):
  def call(self, inputs):
    z_mean, z_log_var = inputs
    batch = tf.shape(z_mean)[0]                                                 # batch = number of data in the batch
    dim = tf.shape(z_mean)[1]                                                   # dim   = number of dimensions of "z"
      # by default, random_normal has mean=0 and std=1.0
    epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
    return z_mean + tf.keras.backend.exp(0.5 * z_log_var) * epsilon



2025-12-05 18:11:29.400979: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-05 18:11:29.825862: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-05 18:11:29.945772: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764969090.028481  154897 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764969090.052237  154897 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764969090.222806  154897 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
import tensorflow as tf
import numpy as np
import inference.fotos as ph
import matplotlib.pyplot as plt


"""
 → mixed_input
(the initial mixture of both sources)

source1_gt → source1_gt
(ground truth image of source 1)

source2_gt → source2_gt
(ground truth image of source 2)

source1_cond → source1_cond
(conditioning vector/label for source 1)

source2_cond → source2_cond
(conditioning vector/label for source 2)

reconstructed_source1 → reconstructed_source1
(filtered estimate of source 1)

reconstructed_source2 → reconstructed_source2
(filtered estimate of source 2)

mask_source1 → mask_source1
(decoder mask/activation applied to mixture for source 1)

mask_source2 → mask_source2
(decoder mask/activation applied to mixture for source 2)

init_placeholder → init_placeholder
(zeros tensor used for initialization)

best_prediction_source1 → best_prediction_source1
(final refined reconstruction of source 1 after evaluation)
"""


class crop:
    def __init__(self, cvae, predictor, data, bias=None, slope=None, **kwargs):
        self.cvae = cvae
        self.predictor = predictor
        self.use_dataset = data
        self.unmix_metrics = {}
        self.reconstruction_metrics = {}
        self.bias = 0.22 if bias is None else bias
        self.slope = 22 if slope is None else slope
        self.beta = 1
        self.alpha_1 = -2
        self.alpha_2 = -22
        self.alpha_mix = 0.5
        self.name = cvae.name

    def best_filtered_var_sigmoid(self, x_mix_filter_2, mixed_input, alpha):
        # First decoded image --------------------------------------------------------------
        x_mix_filter_1 = (
            2 * mixed_input - x_mix_filter_2
        )  # Masked (Cochlear) source1_gt'2
        x_mix_filter_1 = tf.clip_by_value(
            x_mix_filter_1, clip_value_min=0, clip_value_max=1
        )
        condition_encoder = self.predictor.predict(
            x_mix_filter_1, verbose=0
        )  # * j * alfa     # con ponderado incremental

        condition_decoder_1 = condition_encoder

        encoded_imgs = self.cvae.encoder.predict(
            [x_mix_filter_1, condition_encoder], verbose=0
        )

        zz_log_var = encoded_imgs[1] + alpha

        z = Sampling()((encoded_imgs[0], zz_log_var))  # (z_mean, z_log_var)

        mask_source1 = self.cvae.decoder.predict([z, condition_decoder_1], verbose=0)
        mask_source1 = ( mask_source1 - self.bias ) * self.slope 
        mask_source1 = tf.sigmoid(mask_source1)

        x_mix_filter_1 = 2 * mixed_input * mask_source1  # Masked (Cochlear)
        x_mix_filter_1 = tf.clip_by_value(
            x_mix_filter_1, clip_value_min=0, clip_value_max=1
        )

        return (x_mix_filter_1, mask_source1, condition_encoder)

    def graphics(
        self,
        mixed_input,
        source1_gt,
        source2_gt,
        source1_cond,
        source2_cond,
        reconstructed_source1,
        reconstructed_source2,
        mask_source1,
        mask_source2,
        init_placeholder,
        best_prediction_source1,
        bpsnr=None,
        acc_at_least_one=None,
        acc_both=None,
        save_path=None,
    ):

        images = [
            mixed_input,
            source1_gt,
            source2_gt,
            reconstructed_source1,
            reconstructed_source2,
            mask_source1,
            mask_source2,
            init_placeholder,
            best_prediction_source1,
        ]
        row_labels = [
            "x_mix",
            "source2_gt",
            "x_2",
            "x_filt_1",
            "x_filt_2",
            "x_deco_1",
            "x_deco_2",
            "init_placeholder",
            "x_best_pred",
        ]

        num_rows = len(images)
        num_cols = images[0].shape[0] if len(images[0].shape) > 1 else 1
        img_size = 28

        # Figsize proporcional al número de imágenes
        fig_width = num_cols * 1
        fig_height = num_rows * 1
        fig, axes = plt.subplots(num_rows, num_cols, figsize=(fig_width, fig_height))

        # Asegurar que axes siempre sea 2D
        if num_rows == 1 and num_cols == 1:
            axes = np.array([[axes]])
        elif num_rows == 1:
            axes = np.expand_dims(axes, axis=0)
        elif num_cols == 1:
            axes = np.expand_dims(axes, axis=1)

        # ---- Dibujar imágenes ----
        for row in range(num_rows):
            for col in range(num_cols):
                ax = axes[row, col]
                ax.axis("off")

                # Obtener imagen
                img = images[row][col] if num_cols > 1 else images[row]
                if len(img.shape) == 1:
                    img = tf.reshape(img, (img_size, img_size))
                img = img.numpy()
                ax.imshow(img, cmap="gray")
        
                if col == 0:  # solo en la primera columna
                    ax.set_ylabel(
                        row_labels[row],
                        labelpad=40,
                        va="center",
                        rotation=0,
                    )
        # for row, label in enumerate(row_labels):
        #     fig.text(
        #         0.02,  # posición source1_gt relativa
        #         1 - (row + 0.5) / num_rows,  # posición source1_cond relativa
        #         1 - (row + 0.5) / num_rows,
        #         label,
        #         va="center",
        #         ha="right",
        #         fontsize=img_size * 0.4,  # escala con la imagen
        #         rotation=90,
        #     )
        
        # ---- Título arriba con el nombre del modelo ----
        fig.suptitle(self.name, color="darkred")

        # ---- Texto de parámetros abajo ----
        param_text = f"bias={self.bias:.3f}, slope={self.slope:.3f}"
        fig.text(0.5, -0.02, param_text, ha="center", color="darkblue")
        fig.text(
            0.5,
            -0.05,
            f"bpsnr={bpsnr:.3f}, acc_one={acc_at_least_one} acc_both={acc_both}",
            ha="center",
            color="darkblue",
        )
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()

    def unmix(
        self,
        source1_gt,
        source2_gt,
        source1_cond,
        source2_cond,
        iterations=3,
        show_image=False,
        save_path=None,
        curve=False,
        gamma=0.33 # algunos cambios para probar crop2 
    ):
        import inference.metrics as met

        average_image = self.alpha_mix * source1_gt.astype(np.float32) + (
            1 - self.alpha_mix
        ) * source2_gt.astype(np.float32)
        x_mix = average_image

        ## Initialization
        #inicialmente todas las variables son el input.
        mixed_input = x_mix
        mask_source1 = ( x_mix )
        mask_source2 = ( x_mix )
        reconstructed_source1 = ( x_mix )
        reconstructed_source2 = ( x_mix )
        init_placeholder = tf.zeros_like(x_mix)
 
        # condition_encoder = tf.zeros_like(source1_cond)
        acc_at_least_one_plot = []
        acc_both_plot = []
                
        for j in range(iterations):

            reconstructed_source1, mask_source1, predictions_1 = (
                self.best_filtered_var_sigmoid(
                    reconstructed_source2, mixed_input, self.alpha_2
                )
            )

            self.alpha_2 = self.alpha_2 * self.beta

            # algunos cambios para probar crop2 
            x__x = (reconstructed_source1 + reconstructed_source2) / 2 # propuesta-> cambiar por alpha, más logica para decidir cual reconstruccion esta asociada a cada imagen fuente. 

            x__x_e = x__x - x_mix

            reconstructed_source1 = reconstructed_source1 - (x__x_e * gamma)

            reconstructed_source1 = tf.clip_by_value(
                reconstructed_source1, clip_value_min=0, clip_value_max=1
            )

            reconstructed_source2, mask_source2, predictions_2 = (
                self.best_filtered_var_sigmoid(
                    reconstructed_source1, mixed_input, self.alpha_1
                )
            )

            self.alpha_1 = self.alpha_1 * self.beta
            
            # algunos cambios para probar crop2 
            x__x = (reconstructed_source1 + reconstructed_source2) / 2
            x__x_e = x__x - x_mix

            reconstructed_source2 = reconstructed_source2 - (x__x_e * gamma)

            reconstructed_source2 = tf.clip_by_value(
                reconstructed_source2, clip_value_min=0, clip_value_max=1
            )

            # algunos cambios para probar crop2 
            if curve:
                y_predicted_s1_recon = self.predictor.predict(reconstructed_source1, verbose=0)
                y_predicted_s2_recon = self.predictor.predict(reconstructed_source2, verbose=0)
                
                acc_at_least_one, acc_both = met.accuracys(
                p1=y_predicted_s1_recon,
                p2=y_predicted_s2_recon,
                y1=source1_cond,
                y2=source2_cond)

                acc_at_least_one_plot.append(acc_at_least_one)
                acc_both_plot.append(acc_both)
        if curve:
            return{ "acc_at_least_one_plot":acc_at_least_one_plot, "acc_both_plot":acc_both_plot}
        # algunos cambios para probar crop2 

        (
            best_prediction_source1,
            y_predicted_s1_recon,
            y_predicted_s2_recon,
            bpsnr,
            bpsnr_d,
            acc_at_least_one,
            acc_both,
        ) = out.outcomes(
            mask_source1,
            mask_source2,
            reconstructed_source1,
            reconstructed_source2,
            mixed_input,
            source1_gt,
            source2_gt,
            source1_cond,
            source2_cond,
            self.predictor,
        )

        if show_image:
            self.graphics(
                mixed_input,
                source1_gt,
                source2_gt,
                source1_cond,
                source2_cond,
                reconstructed_source1,
                reconstructed_source2,
                mask_source1,
                mask_source2,
                init_placeholder,
                best_prediction_source1,
                bpsnr=bpsnr[0],  # mean value
                acc_at_least_one=acc_at_least_one,
                acc_both=acc_both,
                save_path=save_path,
            )

        return {
            "bpsnr": bpsnr if bpsnr is not None else None, 
            "bpsnr_d": bpsnr_d if bpsnr_d is not None else None,
            "predictions_1": predictions_1 if predictions_1 is not None else None,
            "predictions_2": predictions_2 if predictions_2 is not None else None,
            "acc_at_least_one": acc_at_least_one if acc_at_least_one is not None else None,
            "acc_both": acc_both if acc_both is not None else None,
        }



ModuleNotFoundError: No module named 'project'

In [ ]:





import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
from experiments import load
from CROP_models.crop import crop
import matplotlib.pyplot as plt
#dataset="fashion"
dataset="mnist"

data = load.data(dataset=dataset)
predictor = load.predictor(dataset=dataset)
x_train = data["x_train"]
x_test = data["x_test"]
x_val = data["x_val"]
y_train = data["y_train"]
y_test = data["y_test"]
y_val = data["y_val"]
x_train_1 = data["x_train_1"]
y_train_1 = data["y_train_1"]  
x_test_1  =data["x_test_1"]
y_test_1  =data["y_test_1"]


model = load.cvae(lat=256, inter=512,dataset=dataset)

crop_f0 = crop(model,predictor=predictor,data=data)

iterations=400

metrics  = crop_f0.unmix(
    x_test[:],
    x_test_1[:],
    y_test[:],
    y_test_1[:],
    iterations=iterations,
    curve=True
)


plt.plot(metrics["acc_at_least_one_plot"], label=f"al menos uno ( {metrics["acc_at_least_one_plot"][-1] } )")
plt.plot(metrics["acc_both_plot"], label=f"ambos ({metrics["acc_both_plot"][-1]})")
plt.grid()
plt.title("Accuracy")
plt.xlabel("iterations")
plt.ylabel("acc")
plt.legend()
plt.savefig("acc_at_leat_one_and_both_mnist_default.png")
plt.show()


ImportError: cannot import name 'load' from 'experiments' (/home/santi/Escritorio/tesis/CROP_1/project/experiments/__init__.py)